# RDD2022 Optimize + Train (Kaggle)

This notebook does two things in one run:
1) optimize train split by stratified downsampling (default 1/3)
2) train a strong detector on the optimized dataset


In [ ]:
!pip -q install -U "ultralytics>=8.4.20" "pyyaml>=6.0" "opencv-python-headless>=4.8.0" "tqdm>=4.66"

import csv
import gc
import json
import math
import os
import random
import shutil
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
import yaml
from tqdm.auto import tqdm
from ultralytics import RTDETR

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), "| gpus:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"gpu{i}:", torch.cuda.get_device_name(i))


In [ ]:
# ===== User config =====
DATASET_ROOT_HINT = None              # Example: '/kaggle/input/rdd-2022'
DATASET_HINT_KEYWORD = "rdd-2022"     # keyword for auto-discovery

# Optimize train split
TRAIN_KEEP_RATIO = 0.33               # keep 1/3 of train split
KEEP_VAL_TEST_FULL = True             # keep val/test unchanged
MIN_GROUP_KEEP = 1
CLASS_MIN_KEEP = 120                  # class coverage floor after downsampling

# Train config
MODEL_NAME = "rtdetr-x.pt"          # LOCKED: RT-DETR only
RUN_NAME_BASE = "rdd2022_opt33_rtdetrx_v1"
IMAGE_SIZE = 1024
USE_AMP = False
RUN_PHASE2 = True
PHASE1_EPOCHS = 36
PHASE2_EPOCHS = 12

# Stability/speed
WORKERS = 4
CACHE = False
SAVE_PERIOD = 10
PATIENCE_PHASE1 = 12
PATIENCE_PHASE2 = 8

# Paths
OPT_ROOT = Path("/kaggle/working/input_dataset_opt33")
RUNTIME_YAML = Path("/kaggle/working/data_runtime_opt33.yaml")
PROJECT_DIR = Path("/kaggle/working/runs/detect")
REPORT_JSON = Path("/kaggle/working/report_metrics_opt33.json")
REPORT_CSV = Path("/kaggle/working/report_metrics_opt33.csv")
SUMMARY_JSON = Path("/kaggle/working/dataset_opt_summary.json")

# Device/batch
TRAIN_DEVICE = "0,1" if torch.cuda.is_available() and torch.cuda.device_count() >= 2 else ("0" if torch.cuda.is_available() else "cpu")
EVAL_DEVICE = 0 if torch.cuda.is_available() else "cpu"
BATCH_CANDIDATES = [4, 2, 1] if torch.cuda.device_count() >= 2 else [2, 1]

assert "rtdetr" in MODEL_NAME.lower(), "This notebook is locked to RT-DETR models only."

print("train_device:", TRAIN_DEVICE)
print("eval_device :", EVAL_DEVICE)
print("batch cands :", BATCH_CANDIDATES)


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def read_yaml(path: Path):
    return yaml.safe_load(path.read_text(encoding="utf-8"))


def normalize_names(names_raw):
    if isinstance(names_raw, list):
        return {i: str(v) for i, v in enumerate(names_raw)}
    if isinstance(names_raw, dict):
        out = {}
        for k, v in names_raw.items():
            out[int(k)] = str(v)
        return dict(sorted(out.items(), key=lambda x: x[0]))
    return {0: "crack"}


def find_data_yaml(dataset_root_hint=None, keyword=""):
    if dataset_root_hint:
        p = Path(dataset_root_hint)
        if p.is_file() and p.name == "data.yaml":
            return p.resolve()
        if p.is_dir() and (p / "data.yaml").exists():
            return (p / "data.yaml").resolve()

    cands = []
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not base.exists():
            continue
        for p in base.rglob("data.yaml"):
            s = p.as_posix().lower()
            score = 0
            if keyword and keyword.lower() in s:
                score += 10
            if "rdd" in s:
                score += 3
            if "train" in s or "valid" in s or "val" in s:
                score += 1
            cands.append((score, len(s), p.resolve()))

    if not cands:
        return None
    cands.sort(key=lambda x: (-x[0], x[1]))
    return cands[0][2]


def infer_dataset_root_from_dirs(keyword=""):
    cands = []
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not base.exists():
            continue
        for train_images in base.rglob("train/images"):
            root = train_images.parent.parent
            val_name = None
            if (root / "val" / "images").exists():
                val_name = "val"
            elif (root / "valid" / "images").exists():
                val_name = "valid"
            if val_name is None:
                continue
            s = root.as_posix().lower()
            score = 0
            if keyword and keyword.lower() in s:
                score += 10
            if (root / "test" / "images").exists():
                score += 2
            if (root / "train" / "labels").exists() and (root / val_name / "labels").exists():
                score += 3
            cands.append((score, len(s), root.resolve(), val_name))
    if not cands:
        raise FileNotFoundError("Cannot find dataset root from split dirs")
    cands.sort(key=lambda x: (-x[0], x[1]))
    _, _, root, val_name = cands[0]
    return root, val_name


def resolve_split_path(data_yaml_path, cfg, split_key):
    raw = cfg.get(split_key)
    if raw is None:
        return None
    p = Path(str(raw))
    if p.is_absolute():
        return p

    root = cfg.get("path", None)
    if root:
        root_p = Path(str(root))
        if not root_p.is_absolute():
            root_p = (data_yaml_path.parent / root_p).resolve()
        return (root_p / p).resolve()
    return (data_yaml_path.parent / p).resolve()


def list_images(image_dir: Path):
    if image_dir is None or (not image_dir.exists()):
        return []
    return sorted([x for x in image_dir.rglob("*") if x.is_file() and x.suffix.lower() in IMG_EXTS])


def yolo_seg_to_bbox(coords):
    xs = coords[0::2]
    ys = coords[1::2]
    x1, x2 = max(0.0, min(xs)), min(1.0, max(xs))
    y1, y2 = max(0.0, min(ys)), min(1.0, max(ys))
    w = max(1e-9, x2 - x1)
    h = max(1e-9, y2 - y1)
    return [x1 + w / 2.0, y1 + h / 2.0, w, h]


def normalize_bbox(b):
    cx, cy, w, h = [float(v) for v in b]
    cx = min(1.0, max(0.0, cx))
    cy = min(1.0, max(0.0, cy))
    w = min(1.0, max(1e-9, w))
    h = min(1.0, max(1e-9, h))
    return [cx, cy, w, h]


def parse_label_file(label_path: Path):
    boxes = []
    invalid = 0
    if not label_path.exists():
        return boxes, invalid

    for raw in label_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5:
            invalid += 1
            continue
        try:
            cls = int(float(parts[0]))
            vals = [float(x) for x in parts[1:]]
        except Exception:
            invalid += 1
            continue

        if len(vals) == 4:
            box = normalize_bbox(vals)
        elif len(vals) >= 6 and len(vals) % 2 == 0:
            box = normalize_bbox(yolo_seg_to_bbox(vals))
        else:
            invalid += 1
            continue

        if box[2] <= 1e-9 or box[3] <= 1e-9:
            invalid += 1
            continue
        boxes.append((cls, box[0], box[1], box[2], box[3]))

    # deduplicate
    uniq = {}
    for cls, cx, cy, w, h in boxes:
        key = (int(cls), round(cx, 6), round(cy, 6), round(w, 6), round(h, 6))
        uniq[key] = (int(cls), float(cx), float(cy), float(w), float(h))
    boxes = list(uniq.values())
    return boxes, invalid


def link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
        return
    except Exception:
        pass
    try:
        os.symlink(src, dst)
        return
    except Exception:
        pass
    shutil.copy2(src, dst)


def area_bin(boxes):
    if not boxes:
        return "neg"
    areas = [b[3] * b[4] for b in boxes]
    med = float(np.median(areas))
    if med < 0.0015:
        return "tiny"
    if med < 0.008:
        return "small"
    if med < 0.03:
        return "medium"
    return "large"


def major_class(boxes):
    if not boxes:
        return -1
    c = Counter([b[0] for b in boxes])
    return int(sorted(c.items(), key=lambda x: (-x[1], x[0]))[0][0])


def country_from_name(name: str):
    stem = Path(name).stem
    if "_" in stem:
        token = stem.split("_")[0]
        if token:
            return token
    return "UNK"


In [ ]:
# 1) Discover dataset and split paths

data_yaml = find_data_yaml(DATASET_ROOT_HINT, DATASET_HINT_KEYWORD)
if data_yaml is not None:
    cfg = read_yaml(data_yaml)
    val_key = "val" if "val" in cfg else ("valid" if "valid" in cfg else None)
    if val_key is None:
        raise SystemExit("data.yaml has no val/valid key")
    discover_mode = "from_data_yaml"
else:
    inferred_root, val_key = infer_dataset_root_from_dirs(DATASET_HINT_KEYWORD)
    cfg = {
        "path": str(inferred_root),
        "train": "train/images",
        val_key: f"{val_key}/images",
    }
    if (inferred_root / "test" / "images").exists():
        cfg["test"] = "test/images"
    data_yaml = inferred_root / "data.yaml"
    discover_mode = "from_split_dirs"

split_src = {
    "train": resolve_split_path(data_yaml, cfg, "train"),
    "val": resolve_split_path(data_yaml, cfg, val_key),
    "test": resolve_split_path(data_yaml, cfg, "test") if "test" in cfg else None,
}

for req in ["train", "val"]:
    p = split_src.get(req)
    if p is None or (not p.exists()):
        raise SystemExit(f"Missing split {req}: {p}")
    if len(list_images(p)) == 0:
        raise SystemExit(f"No images in split {req}: {p}")

names_map = normalize_names(cfg.get("names", {0: "crack"}))
nc = int(cfg.get("nc", len(names_map)))
if nc != len(names_map):
    nc = len(names_map)

print("discover_mode:", discover_mode)
print("source data_yaml:", data_yaml)
print("split_src:", {k: (str(v) if v else None) for k, v in split_src.items()})
print("nc:", nc)
print("names:", names_map)

# 2) Build train metadata
train_img_dir = split_src["train"]
train_lbl_dir = train_img_dir.parent / "labels"
train_images = list_images(train_img_dir)

train_items = []
class_to_items = defaultdict(list)
for img_path in tqdm(train_images, desc="scan train"):
    rel = img_path.relative_to(train_img_dir)
    lbl = train_lbl_dir / rel.with_suffix(".txt")
    boxes, invalid = parse_label_file(lbl)
    mcls = major_class(boxes)
    abin = area_bin(boxes)
    ctry = country_from_name(rel.name)
    key = (str(mcls), abin, ctry)
    item = {
        "rel": rel.as_posix(),
        "has_obj": len(boxes) > 0,
        "boxes": boxes,
        "invalid": int(invalid),
        "major_class": int(mcls),
        "area_bin": abin,
        "country": ctry,
        "strata": key,
        "classes": sorted(list(set([b[0] for b in boxes]))),
    }
    idx = len(train_items)
    train_items.append(item)
    for cid in item["classes"]:
        class_to_items[int(cid)].append(idx)

# 3) Stratified downsample train
rng = random.Random(SEED)
strata_to_indices = defaultdict(list)
for i, it in enumerate(train_items):
    strata_to_indices[it["strata"]].append(i)

selected = set()
for strata, idxs in strata_to_indices.items():
    n = len(idxs)
    k = max(MIN_GROUP_KEEP, int(round(n * float(TRAIN_KEEP_RATIO))))
    if k >= n:
        selected.update(idxs)
    else:
        chosen = rng.sample(idxs, k)
        selected.update(chosen)

# Class coverage booster
for cid in range(nc):
    all_ids = class_to_items.get(cid, [])
    if not all_ids:
        continue
    cur = sum(1 for x in all_ids if x in selected)
    target = min(len(all_ids), max(CLASS_MIN_KEEP, int(round(len(all_ids) * float(TRAIN_KEEP_RATIO) * 0.7))))
    if cur < target:
        pool = [x for x in all_ids if x not in selected]
        need = min(len(pool), target - cur)
        if need > 0:
            selected.update(rng.sample(pool, need))

selected_rels = set([train_items[i]["rel"] for i in selected])

# 4) Materialize optimized dataset
if OPT_ROOT.exists():
    shutil.rmtree(OPT_ROOT)
(OPT_ROOT / "train" / "images").mkdir(parents=True, exist_ok=True)
(OPT_ROOT / "train" / "labels").mkdir(parents=True, exist_ok=True)

summary = {
    "source": {
        "discover_mode": discover_mode,
        "data_yaml": str(data_yaml),
        "split_paths": {k: (str(v) if v else None) for k, v in split_src.items()},
        "nc": nc,
        "names": {int(k): str(v) for k, v in names_map.items()},
    },
    "opt": {
        "train_keep_ratio": float(TRAIN_KEEP_RATIO),
        "selected_train_images": int(len(selected_rels)),
        "source_train_images": int(len(train_images)),
    },
    "splits": {}
}


def write_split(split_name, src_img_dir, selected_rel_set=None):
    if src_img_dir is None or (not src_img_dir.exists()):
        return None
    src_lbl_dir = src_img_dir.parent / "labels"
    out_img_dir = OPT_ROOT / split_name / "images"
    out_lbl_dir = OPT_ROOT / split_name / "labels"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    images = list_images(src_img_dir)
    if selected_rel_set is not None:
        images = [p for p in images if p.relative_to(src_img_dir).as_posix() in selected_rel_set]

    pos_images = 0
    neg_images = 0
    invalid_lines = 0
    box_count = 0
    class_counter = Counter()

    for img_path in tqdm(images, desc=f"write {split_name}"):
        rel = img_path.relative_to(src_img_dir)
        dst_img = out_img_dir / rel
        dst_lbl = out_lbl_dir / rel.with_suffix(".txt")
        link_or_copy(img_path, dst_img)

        src_lbl = src_lbl_dir / rel.with_suffix(".txt")
        boxes, bad = parse_label_file(src_lbl)
        invalid_lines += int(bad)

        if boxes:
            pos_images += 1
            box_count += len(boxes)
            lines = []
            for cls, cx, cy, w, h in boxes:
                class_counter[int(cls)] += 1
                lines.append(f"{int(cls)} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            dst_lbl.write_text("\n".join(lines) + "\n", encoding="utf-8")
        else:
            neg_images += 1
            dst_lbl.write_text("", encoding="utf-8")

    return {
        "images": int(len(images)),
        "pos_images": int(pos_images),
        "neg_images": int(neg_images),
        "boxes": int(box_count),
        "invalid_label_lines": int(invalid_lines),
        "class_boxes": {int(k): int(v) for k, v in sorted(class_counter.items())},
    }

summary["splits"]["train"] = write_split("train", split_src["train"], selected_rels)
summary["splits"]["val"] = write_split("val", split_src["val"], None)
if KEEP_VAL_TEST_FULL and split_src.get("test") is not None:
    summary["splits"]["test"] = write_split("test", split_src["test"], None)

runtime = {
    "path": str(OPT_ROOT),
    "train": "train/images",
    "val": "val/images",
    "nc": int(nc),
    "names": {int(k): str(v) for k, v in sorted(names_map.items(), key=lambda x: int(x[0]))},
}
if (OPT_ROOT / "test" / "images").exists():
    runtime["test"] = "test/images"

RUNTIME_YAML.write_text(yaml.safe_dump(runtime, sort_keys=False, allow_unicode=False), encoding="utf-8")
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Runtime YAML:", RUNTIME_YAML)
print("Summary JSON:", SUMMARY_JSON)
print(json.dumps(summary, indent=2))

if summary["splits"]["train"]["images"] == 0:
    raise SystemExit("Optimized train split is empty")
if summary["splits"]["val"]["images"] == 0:
    raise SystemExit("Optimized val split is empty")
if sum(summary["splits"]["train"]["class_boxes"].values()) == 0:
    raise SystemExit("No positive labels left in optimized train split")

print("DATASET OPTIMIZE PASS")


In [ ]:
def empty_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



def new_model(model_source: str):
    return RTDETR(model_source)


def train_with_batch_fallback(model_source, run_name, train_args, batch_candidates):
    last_error = None
    for b in batch_candidates:
        empty_cache()
        t0 = time.time()
        print(f"\nStart {run_name} | model={model_source} | batch={b}")
        try:
            model = new_model(model_source)
            model.train(name=run_name, batch=int(b), **train_args)
            elapsed = (time.time() - t0) / 60.0
            save_dir = Path(train_args["project"]) / run_name
            return {
                "ok": True,
                "model_type": "rtdetr",
                "batch": int(b),
                "elapsed_min": float(elapsed),
                "save_dir": str(save_dir),
                "error": None,
            }
        except Exception as e:
            last_error = e
            msg = str(e).lower()
            print(f"[train] batch={b} failed: {e}")
            if ("out of memory" in msg) or ("cuda error" in msg) or ("cudnn" in msg):
                continue
            raise
    raise RuntimeError(f"All batch candidates failed. Last error: {last_error}")


def check_results_health(run_dir: Path):
    csv_path = run_dir / "results.csv"
    if not csv_path.exists():
        return
    with csv_path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return
    bad = []
    for i, row in enumerate(rows, start=1):
        for k, v in row.items():
            if "loss" not in k.lower():
                continue
            try:
                fv = float(v)
            except Exception:
                continue
            if not np.isfinite(fv):
                bad.append((i, k, v))
                if len(bad) >= 5:
                    break
        if len(bad) >= 5:
            break
    if bad:
        sample = "; ".join([f"row={r} {k}={v}" for r, k, v in bad])
        raise RuntimeError(f"NaN/Inf found in results.csv: {sample}")


common_train_args = dict(
    data=str(RUNTIME_YAML),
    device=TRAIN_DEVICE,
    imgsz=IMAGE_SIZE,
    optimizer="AdamW",
    weight_decay=0.0005,
    warmup_epochs=3.0,
    cos_lr=True,
    pretrained=True,
    amp=bool(USE_AMP),
    cache=bool(CACHE),
    workers=int(WORKERS),
    save=True,
    save_period=int(SAVE_PERIOD),
    plots=True,
    val=True,
    verbose=True,
    project=str(PROJECT_DIR),
    exist_ok=True,
    seed=SEED,
    deterministic=False,
)

phase1_args = dict(common_train_args)
phase1_args.update(dict(
    epochs=int(PHASE1_EPOCHS),
    lr0=0.0002,
    lrf=0.01,
    patience=int(PATIENCE_PHASE1),
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=2.0,
    translate=0.08,
    scale=0.35,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.20,
    mixup=0.03,
    close_mosaic=12,
    erasing=0.10,
))

phase1_name = f"{RUN_NAME_BASE}_phase1"
phase1 = train_with_batch_fallback(MODEL_NAME, phase1_name, phase1_args, BATCH_CANDIDATES)
phase1_dir = Path(phase1["save_dir"])
check_results_health(phase1_dir)
phase1_best = phase1_dir / "weights" / "best.pt"
phase1_last = phase1_dir / "weights" / "last.pt"
assert phase1_best.exists(), f"Missing phase1 best: {phase1_best}"

phase2 = None
phase2_dir = None
final_best = phase1_best
final_last = phase1_last

if RUN_PHASE2 and int(PHASE2_EPOCHS) > 0:
    phase2_args = dict(common_train_args)
    phase2_args.update(dict(
        epochs=int(PHASE2_EPOCHS),
        lr0=0.00006,
        lrf=0.05,
        patience=int(PATIENCE_PHASE2),
        hsv_h=0.010,
        hsv_s=0.35,
        hsv_v=0.2,
        degrees=0.5,
        translate=0.02,
        scale=0.15,
        fliplr=0.5,
        flipud=0.0,
        mosaic=0.0,
        mixup=0.0,
        close_mosaic=1,
        erasing=0.0,
        save_period=0,
    ))

    phase2_name = f"{RUN_NAME_BASE}_phase2"
    phase2 = train_with_batch_fallback(str(phase1_best), phase2_name, phase2_args, BATCH_CANDIDATES)
    phase2_dir = Path(phase2["save_dir"])
    check_results_health(phase2_dir)
    final_best = phase2_dir / "weights" / "best.pt"
    final_last = phase2_dir / "weights" / "last.pt"
    assert final_best.exists(), f"Missing final best: {final_best}"

print("phase1:", phase1)
print("phase2:", phase2)
print("final_best:", final_best)


In [ ]:
def pack_metrics(metrics):
    return {
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    }

empty_cache()
model_eval = RTDETR(str(final_best))
val_metrics = model_eval.val(data=str(RUNTIME_YAML), split="val", device=EVAL_DEVICE, imgsz=IMAGE_SIZE, batch=1, verbose=True)

runtime_cfg = read_yaml(RUNTIME_YAML)
test_pack = None
if "test" in runtime_cfg:
    test_metrics = model_eval.val(data=str(RUNTIME_YAML), split="test", device=EVAL_DEVICE, imgsz=IMAGE_SIZE, batch=1, verbose=True)
    test_pack = pack_metrics(test_metrics)

report = {
    "model_name": MODEL_NAME,
    "run_name_base": RUN_NAME_BASE,
    "image_size": IMAGE_SIZE,
    "train_device": TRAIN_DEVICE,
    "batch_candidates": BATCH_CANDIDATES,
    "phase1_epochs": int(PHASE1_EPOCHS),
    "phase2_epochs": int(PHASE2_EPOCHS if RUN_PHASE2 else 0),
    "phase1": phase1,
    "phase2": phase2,
    "phase1_best": str(phase1_best),
    "final_best": str(final_best),
    "val": pack_metrics(val_metrics),
    "test": test_pack,
    "dataset_summary": summary,
}

REPORT_JSON.write_text(json.dumps(report, indent=2), encoding="utf-8")
with REPORT_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["split", "mAP50", "mAP50-95", "precision", "recall"])
    writer.writeheader()
    writer.writerow({"split": "val", **report["val"]})
    if report["test"]:
        writer.writerow({"split": "test", **report["test"]})

print(json.dumps(report, indent=2))
print("Saved report:", REPORT_JSON)


In [ ]:
bundle = Path("/kaggle/working") / f"{RUN_NAME_BASE}_bundle.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [RUNTIME_YAML, REPORT_JSON, REPORT_CSV, SUMMARY_JSON]:
        if p.exists():
            zf.write(p, arcname=p.name)
    for p in [phase1_best, phase1_last, final_best, final_last]:
        if p.exists():
            zf.write(p, arcname=f"weights/{p.name}")

    run_dirs = [(phase1_dir, "phase1")]
    if phase2_dir is not None:
        run_dirs.append((phase2_dir, "phase2"))
    for run_dir, tag in run_dirs:
        for rel in ["args.yaml", "results.csv", "results.png", "PR_curve.png", "P_curve.png", "R_curve.png", "F1_curve.png", "confusion_matrix.png"]:
            p = run_dir / rel
            if p.exists():
                zf.write(p, arcname=f"{tag}/{rel}")

print("Bundle:", bundle)
print("Bundle MB:", round(bundle.stat().st_size / (1024 * 1024), 2))
